In [ ]:
# pip install -U google-generativeai gradio pandas langchain langchain-community huggingface-hub
# If you're in a notebook environment, you may prefer:
# %pip install -U google-generativeai gradio pandas langchain langchain-community huggingface-hub

%matplotlib inline


# Task 5 — Auto-Tagging Support Tickets (LLM Prompting)

## Objective
Automatically tag support tickets into one of 5 categories using **prompt engineering** (no fine-tuning):
`["billing", "technical", "account", "product", "shipping"]`.

## Dataset
A small synthetic dataset is created in this notebook (20 labeled tickets saved as a CSV). A separate labeled set of 10 test tickets is used for a quick evaluation.

## LLM
- Primary: **Gemini 2.5 Flash** via `google-generativeai` (free tier)
- Fallback: Hugging Face free inference via `HuggingFaceEndpoint` with `HF_TOKEN` (e.g., `mistralai/Mistral-7B-Instruct-v0.3`)

## Approach
- **Zero-shot**: ask the LLM to return top-3 tags with confidences in JSON
- **Few-shot**: provide labeled examples (3 per category) in the prompt
- Evaluate on 10 labeled test tickets (approximate accuracy)
- Build a Gradio demo to tag new tickets

## Final Summary / Insights
Few-shot prompts typically improve consistency of labeling and JSON formatting compared to zero-shot prompts, especially on edge-case tickets.


In [ ]:
import json
import os
import re
from typing import Dict, List, Optional, Tuple

import pandas as pd

try:
    import google.generativeai as genai
except Exception:
    genai = None

try:
    import gradio as gr
except Exception as e:
    raise ImportError("gradio is required. Install with: pip install gradio") from e

try:
    from langchain_community.llms import HuggingFaceEndpoint
except Exception as e:
    raise ImportError(
        "langchain-community is required for HuggingFaceEndpoint fallback. Install with: pip install langchain-community"
    ) from e

LABELS = ["billing", "technical", "account", "product", "shipping"]
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")


In [ ]:
# Configure LLM (Gemini primary, HF fallback)
def build_gemini_model():
    if not genai or not GOOGLE_API_KEY:
        return None
    try:
        genai.configure(api_key=GOOGLE_API_KEY)
        for name in ["gemini-2.5-flash", "gemini-1.5-flash"]:
            try:
                return genai.GenerativeModel(name)
            except Exception:
                continue
        return None
    except Exception:
        return None

def build_hf_endpoint():
    if not HF_TOKEN:
        return None
    try:
        return HuggingFaceEndpoint(
            repo_id="mistralai/Mistral-7B-Instruct-v0.3",
            huggingfacehub_api_token=HF_TOKEN,
            temperature=0.0,
            max_new_tokens=256,
        )
    except Exception:
        return None

gemini_model = build_gemini_model()
hf_llm = build_hf_endpoint()

if gemini_model:
    print("Using Gemini.")
elif hf_llm:
    print("Using Hugging Face endpoint fallback.")
else:
    print(
        "No LLM configured. Set GOOGLE_API_KEY for Gemini, or HF_TOKEN for Hugging Face fallback, then re-run this cell."
    )

def call_llm(prompt: str) -> str:
    if gemini_model:
        try:
            resp = gemini_model.generate_content(prompt)
            return (resp.text or "").strip()
        except Exception as e:
            return f"Gemini call failed: {e}"
    if hf_llm:
        try:
            return str(hf_llm.invoke(prompt)).strip()
        except Exception as e:
            return f"Hugging Face endpoint call failed: {e}"
    return "No LLM configured. Set GOOGLE_API_KEY or HF_TOKEN and re-run configuration cells."


In [ ]:
# Create a small labeled dataset (20 examples) and save to CSV
train_rows = [
    {"text": "I was charged twice for my subscription this month.", "label": "billing"},
    {"text": "Can you refund the extra fee on my last invoice?", "label": "billing"},
    {"text": "My discount code didn't apply at checkout.", "label": "billing"},
    {"text": "The app crashes whenever I try to upload a file.", "label": "technical"},
    {"text": "I'm seeing a 500 error when I open the dashboard.", "label": "technical"},
    {"text": "Push notifications stopped working after the update.", "label": "technical"},
    {"text": "I can't log in even after resetting my password.", "label": "account"},
    {"text": "Please change the email address on my account.", "label": "account"},
    {"text": "My account was locked due to suspicious activity.", "label": "account"},
    {"text": "Does the pro plan include team collaboration features?", "label": "product"},
    {"text": "Where can I find the user guide for the new feature?", "label": "product"},
    {"text": "Feature X is missing in my workspace after upgrading.", "label": "product"},
    {"text": "My order hasn't arrived and tracking hasn't updated.", "label": "shipping"},
    {"text": "The package was marked delivered but I didn't receive it.", "label": "shipping"},
    {"text": "Can you change my delivery address for the shipment?", "label": "shipping"},
    {"text": "I need an invoice for last quarter's payments.", "label": "billing"},
    {"text": "The website is very slow and times out frequently.", "label": "technical"},
    {"text": "How do I delete my account permanently?", "label": "account"},
    {"text": "Is there an API for integrating this product with Slack?", "label": "product"},
    {"text": "My replacement item shipment is delayed again.", "label": "shipping"},
]

df_train = pd.DataFrame(train_rows)
CSV_PATH = "./task5_support_tickets_20.csv"
df_train.to_csv(CSV_PATH, index=False)
print("Saved:", CSV_PATH)
display(df_train.head())


In [ ]:
# Labeled test set (10 examples) for quick evaluation
test_rows = [
    {"text": "Why did my plan renew at a higher price this month?", "label": "billing"},
    {"text": "The mobile app freezes on the login screen.", "label": "technical"},
    {"text": "I want to update my password but the reset link is expired.", "label": "account"},
    {"text": "Does your product support single sign-on (SSO)?", "label": "product"},
    {"text": "Tracking says in transit for 10 days. Any update?", "label": "shipping"},
    {"text": "I was billed after canceling my subscription.", "label": "billing"},
    {"text": "Uploads fail with a network error even on Wi-Fi.", "label": "technical"},
    {"text": "Please help me recover my account after losing 2FA device.", "label": "account"},
    {"text": "Can I export my data from the product to CSV?", "label": "product"},
    {"text": "My package arrived damaged. What are the next steps?", "label": "shipping"},
]
df_test = pd.DataFrame(test_rows)
display(df_test)


In [ ]:
JSON_SCHEMA = {
    "predictions": [
        {"label": "billing", "confidence": 0.67},
        {"label": "account", "confidence": 0.20},
        {"label": "technical", "confidence": 0.13},
    ]
}

def parse_predictions(text: str) -> Dict[str, object]:
    # Extract the first JSON object if the model includes extra text.
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        return {"error": "No JSON found in model output", "raw": text}
    try:
        obj = json.loads(match.group(0))
    except Exception as e:
        return {"error": f"Invalid JSON: {e}", "raw": text}

    preds = obj.get("predictions")
    if not isinstance(preds, list):
        return {"error": "JSON missing 'predictions' list", "raw": text, "json": obj}

    clean = []
    for p in preds[:3]:
        if not isinstance(p, dict):
            continue
        label = str(p.get("label", "")).strip().lower()
        conf = p.get("confidence")
        if label not in LABELS:
            continue
        try:
            conf_f = float(conf)
        except Exception:
            continue
        conf_f = max(0.0, min(1.0, conf_f))
        clean.append({"label": label, "confidence": conf_f})

    if not clean:
        return {"error": "No valid predictions parsed", "raw": text, "json": obj}

    clean = sorted(clean, key=lambda d: d["confidence"], reverse=True)[:3]
    return {"predictions": clean}

def build_zero_shot_prompt(ticket: str) -> str:
    return (
        "You are a support ticket triage assistant. "
        f"Possible labels: {LABELS}.\n\n"
        "Given the ticket text, output ONLY valid JSON with this schema:\n"
        f"{json.dumps(JSON_SCHEMA, indent=2)}\n\n"
        "Rules:\n"
        "- Return exactly 3 predictions sorted by confidence (descending)\n"
        "- confidence must be a number between 0 and 1\n"
        "- Use labels exactly as provided\n\n"
        f"TICKET: {ticket}"
    )

def build_few_shot_prompt(ticket: str, examples: pd.DataFrame) -> str:
    ex_lines = []
    for _, row in examples.iterrows():
        ex_lines.append(f"Ticket: {row['text']}\nLabel: {row['label']}\n")
    examples_block = "\n".join(ex_lines)
    return (
        "You are a support ticket triage assistant. "
        f"Possible labels: {LABELS}.\n\n"
        "Here are labeled examples:\n"
        f"{examples_block}\n"
        "Now, for the new ticket, output ONLY valid JSON with this schema:\n"
        f"{json.dumps(JSON_SCHEMA, indent=2)}\n\n"
        "Rules:\n"
        "- Return exactly 3 predictions sorted by confidence (descending)\n"
        "- confidence must be a number between 0 and 1\n"
        "- Use labels exactly as provided\n\n"
        f"TICKET: {ticket}"
    )


In [ ]:
# Few-shot examples: 3 per category (15 total)
few_shot_examples = (
    df_train.groupby("label", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)
few_shot_examples["label"].value_counts()


In [ ]:
def predict_tags(ticket: str, mode: str) -> Dict[str, object]:
    if not ticket or not ticket.strip():
        return {"error": "Empty ticket"}

    if mode == "few-shot":
        prompt = build_few_shot_prompt(ticket.strip(), few_shot_examples)
    else:
        prompt = build_zero_shot_prompt(ticket.strip())

    raw = call_llm(prompt)
    parsed = parse_predictions(raw)
    if "error" in parsed:
        return {"error": parsed["error"], "raw": raw}
    return parsed

def top1_label(pred: Dict[str, object]) -> Optional[str]:
    preds = pred.get("predictions")
    if not isinstance(preds, list) or not preds:
        return None
    return str(preds[0].get("label"))

def top3_labels(pred: Dict[str, object]) -> List[str]:
    preds = pred.get("predictions")
    if not isinstance(preds, list):
        return []
    return [str(p.get("label")) for p in preds[:3] if isinstance(p, dict)]


In [ ]:
# Evaluate on 10 test tickets
rows = []
for _, r in df_test.iterrows():
    ticket = r["text"]
    true_label = r["label"]

    zs = predict_tags(ticket, mode="zero-shot")
    fs = predict_tags(ticket, mode="few-shot")

    rows.append(
        {
            "text": ticket,
            "true": true_label,
            "zero_top1": top1_label(zs),
            "zero_top3": top3_labels(zs),
            "few_top1": top1_label(fs),
            "few_top3": top3_labels(fs),
        }
    )

df_eval = pd.DataFrame(rows)
df_eval["zero_top1_correct"] = df_eval["zero_top1"] == df_eval["true"]
df_eval["few_top1_correct"] = df_eval["few_top1"] == df_eval["true"]
df_eval["zero_top3_correct"] = df_eval.apply(lambda r: r["true"] in (r["zero_top3"] or []), axis=1)
df_eval["few_top3_correct"] = df_eval.apply(lambda r: r["true"] in (r["few_top3"] or []), axis=1)

print("Zero-shot Top-1 accuracy:", df_eval["zero_top1_correct"].mean())
print("Few-shot Top-1 accuracy:", df_eval["few_top1_correct"].mean())
print("Zero-shot Top-3 accuracy:", df_eval["zero_top3_correct"].mean())
print("Few-shot Top-3 accuracy:", df_eval["few_top3_correct"].mean())

display(df_eval[["true", "zero_top1", "zero_top3", "few_top1", "few_top3"]])


## Gradio Demo
Type a support ticket and choose **zero-shot** or **few-shot** tagging. The output is JSON with the top-3 predicted tags.


In [ ]:
def gradio_predict(ticket: str, mode: str):
    mode_key = "few-shot" if mode.lower().startswith("few") else "zero-shot"
    return predict_tags(ticket, mode=mode_key)

demo = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Textbox(lines=4, label="Support Ticket"),
        gr.Radio(["zero-shot", "few-shot"], value="few-shot", label="Mode"),
    ],
    outputs=gr.JSON(label="Top-3 Tags"),
    title="Support Ticket Auto-Tagging (LLM Prompting)",
    description=(
        "Uses Gemini if GOOGLE_API_KEY is set, otherwise falls back to a free Hugging Face endpoint if HF_TOKEN is set. "
        "Returns top-3 tags as JSON."
    ),
    examples=[
        ["I can't access my account after changing my phone number.", "few-shot"],
        ["My order says delivered but I never got the package.", "few-shot"],
        ["I was billed even though I canceled yesterday.", "zero-shot"],
    ],
)

demo.launch()
